In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By        # 요소 찾기 기준(By.ID, By.NAME ...)
from selenium.webdriver.common.keys import Keys     # 키보드 입력(Enter 등)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import time

driver = webdriver.Chrome()
driver.get("https://www.google.com")
driver.implicitly_wait(10)              # 전역 암묵적 대기

search_box = driver.find_element("name", "q")   # 검색창(name='q')
search_box.send_keys("날씨")                     # 글자 입력
search_box.send_keys(Keys.RETURN)               # 엔터로 전송
time.sleep(3)
# driver.quit()                                   # 브라우저 종료(필수)

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/dynamic_controls")

# 'Enable' 버튼 클릭 → 입력창 활성화
driver.find_element(By.XPATH, "//button[text()='Enable']").click()

# 입력창이 클릭 가능해질 때까지 최대 10초 대기
input_field = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//input[@type='text']"))
)
input_field.send_keys("테스트 입력 완료!")
print("✅ 활성화 후 입력 성공")
time.sleep(3)
driver.quit()

✅ 활성화 후 입력 성공


In [5]:
from selenium import webdriver
import time
driver = webdriver.Chrome()
driver.get("https://news.google.com")
time.sleep(3)

# 맨 아래로 스크롤
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

# 일정 크기만큼 스크롤하며 캐폐
for i in range(3):
    driver.save_screenshot(f"news_{i+1}.png")
    driver.execute_script("window.scrollBy(0, 600);")   # 600px씩
    time.sleep(2)

# 쿠키 추가 · 확인
driver.add_cookie({"name": "lunch", "value": "신라면"})
driver.refresh(); time.sleep(2)
for c in driver.get_cookies():
    if c["name"] == "lunch":
        print("쿠키 값:", c["value"])
driver.quit()

쿠키 값: 신라면


In [6]:
import time, json
from bs4 import BeautifulSoup
from selenium import webdriver

def coffeebean_stores(max_no=10):
    url = "https://www.coffeebeankorea.com/store/store.asp"
    wd = webdriver.Chrome()
    result = []
    for i in range(1, max_no + 1):      # 1~max_no 매장 (전체는 큰 수 사용)
        wd.get(url); time.sleep(1)
        try:
            wd.execute_script(f"storePop2({i})")   # 사이트 내부 JS 함수 호출
            time.sleep(2)
            soup = BeautifulSoup(wd.page_source, 'html.parser')
            name = soup.select("div.store_txt > h2")
            name = name[0].get_text(strip=True) if name else "정보 없음"
            tds = soup.select("div.store_txt > table.store_table > tbody > tr > td")
            addr  = tds[2].get_text(strip=True) if len(tds) > 2 else "주소 없음"
            phone = tds[3].get_text(strip=True) if len(tds) > 3 else "전화 없음"
            print(f"📍 {name} | {addr} | {phone}")
            result.append({"store": name, "address": addr, "phone": phone})
        except Exception as e:
            print("❌ 오류:", e)
    wd.quit()
    return result

data = coffeebean_stores(10)
with open("CoffeeBean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)
print("✅ JSON 저장 완료:", len(data), "개")

📍 정보 없음 | 주소 없음 | 전화 없음
📍 정보 없음 | 주소 없음 | 전화 없음
📍 차병원점 | 서울시 강남구 논현로 566 강남차병원1층 | 02-538-7615
📍 정보 없음 | 주소 없음 | 전화 없음
📍 정보 없음 | 주소 없음 | 전화 없음
📍 강남대로점 | 서울시 서초구 강남대로 369 1층 | 02-588-5778
📍 정보 없음 | 주소 없음 | 전화 없음
📍 정보 없음 | 주소 없음 | 전화 없음
📍 정보 없음 | 주소 없음 | 전화 없음
📍 정보 없음 | 주소 없음 | 전화 없음
✅ JSON 저장 완료: 10 개


In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time, os, urllib.request

query = "맛조개"
save_path = os.path.join(os.getcwd(), query)   # OS 공통 경로 (윈도우/맥 모두 OK)
os.makedirs(save_path, exist_ok=True)

driver = webdriver.Chrome()
driver.get("https://www.google.com/imghp")
bar = driver.find_element(By.NAME, "q")
bar.send_keys(query); bar.submit()
time.sleep(60)

# 더 이상 길이가 안 늘어날 때까지 스크롤 (무한스크롤)
prev = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(2)
    curr = driver.execute_script("return document.body.scrollHeight")
    if curr == prev:
        break
    prev = curr

imgs = driver.find_elements(By.TAG_NAME, "img")
for idx, img in enumerate(imgs):
    try:
        src = img.get_attribute("src")
        if src and src.startswith("http"):
            urllib.request.urlretrieve(src, f"{save_path}/{query}_{idx}.png")
            print(f"✅ {idx} 다운로드 완료")
    except Exception as e:
        print(f"❌ {idx} 실패: {e}")
driver.quit()
print("done")

✅ 0 다운로드 완료
✅ 12 다운로드 완료
✅ 13 다운로드 완료
✅ 14 다운로드 완료
✅ 34 다운로드 완료
✅ 212 다운로드 완료
✅ 214 다운로드 완료
✅ 216 다운로드 완료
✅ 218 다운로드 완료
✅ 220 다운로드 완료
✅ 222 다운로드 완료
✅ 224 다운로드 완료
✅ 226 다운로드 완료
✅ 228 다운로드 완료
✅ 230 다운로드 완료
✅ 232 다운로드 완료
✅ 234 다운로드 완료
✅ 236 다운로드 완료
✅ 237 다운로드 완료
✅ 238 다운로드 완료
✅ 239 다운로드 완료
✅ 241 다운로드 완료
✅ 243 다운로드 완료
✅ 245 다운로드 완료
✅ 247 다운로드 완료
✅ 248 다운로드 완료
✅ 249 다운로드 완료
✅ 250 다운로드 완료
✅ 251 다운로드 완료
✅ 252 다운로드 완료
✅ 253 다운로드 완료
✅ 254 다운로드 완료
✅ 255 다운로드 완료
✅ 256 다운로드 완료
✅ 257 다운로드 완료
✅ 258 다운로드 완료
✅ 259 다운로드 완료
✅ 260 다운로드 완료
✅ 261 다운로드 완료
✅ 262 다운로드 완료
✅ 263 다운로드 완료
✅ 264 다운로드 완료
✅ 265 다운로드 완료
✅ 266 다운로드 완료
✅ 267 다운로드 완료
✅ 268 다운로드 완료
✅ 269 다운로드 완료
✅ 270 다운로드 완료
✅ 271 다운로드 완료
✅ 272 다운로드 완료
✅ 273 다운로드 완료
✅ 274 다운로드 완료
✅ 275 다운로드 완료
✅ 276 다운로드 완료
✅ 277 다운로드 완료
✅ 278 다운로드 완료
✅ 279 다운로드 완료
✅ 280 다운로드 완료
✅ 281 다운로드 완료
✅ 282 다운로드 완료
✅ 283 다운로드 완료
✅ 284 다운로드 완료
✅ 285 다운로드 완료
✅ 286 다운로드 완료
✅ 287 다운로드 완료
✅ 288 다운로드 완료
✅ 289 다운로드 완료
✅ 290 다운로드 완료
✅ 291 다운로드 완료
✅ 292 다운로드 완료
✅ 293 다운로드 완료
✅ 294 다운로드 완

KeyboardInterrupt: 